## K-Nearest Neighbors (KNN)

O [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) é um algoritmo baseado em instâncias, que classifica uma nova observação com base nos seus vizinhos mais próximos.

A classe é definida pela maioria dos vizinhos.

### Hiperparâmetros utilizados

- **n_neighbors**: número de vizinhos considerados (default = 5)
  - Valores menores → modelo mais sensível 
  - Valores maiores → modelo mais estável

- **weights**: forma de ponderação
  - `uniform`: todos os vizinhos têm o mesmo peso (uniform)
  - `distance`: vizinhos mais próximos têm maior peso

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para encontrar os melhores parâmetros, utilizando ROC AUC como métrica.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []
#         steps.append(('scaler', StandardScaler()))
#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('knn', KNeighborsClassifier()))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "KNN",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })

# save_results(results, file_path="../results/experiments.csv")

# df_result = pd.DataFrame(results)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )
    print(f"Colunas utilizadas para o cenario '{scenario}': {X_train.columns.tolist()}")
    print(f"Total de features: {len(X_train.columns)}")


    param_grid_knn = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "knn__n_neighbors": [3, 5, 7, 9],
        "knn__weights": ["uniform", "distance"],
        "knn__metric": ["euclidean", "manhattan"]
    }

    grid_knn = GridSearchCV(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('knn', KNeighborsClassifier())
        ]),
        param_grid=param_grid_knn,
        cv=5,
        scoring="roc_auc",
        n_jobs=2
    )

    grid_knn.fit(X_train, y_train)

    print("Best parameters:", grid_knn.best_params_)
    print("Best ROC AUC (CV):", grid_knn.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_knn = grid_knn.best_estimator_

    y_pred = best_knn.predict(X_test)
    y_proba = best_knn.predict_proba(X_test)[:, 1]


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_knn.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "KNN",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_knn.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "KNN",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Colunas utilizadas para o cenario 'sem_submodalidade': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários mínimos', '

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.709504,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:03:54.190012
1,KNN,sem_submodalidade,True,test,0.720504,0.712963,0.670031,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.808451
2,KNN,sem_submodalidade,False,tuning_cv,0.714070,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.809167
3,KNN,sem_submodalidade,False,test,0.726809,0.725928,0.676821,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:35.312272



🔎 Scenario: submodalidade_agrupada
Colunas utilizadas para o cenario 'submodalidade_agrupada': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários m

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.709504,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:03:54.190012
1,KNN,sem_submodalidade,True,test,0.720504,0.712963,0.670031,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.808451
2,KNN,sem_submodalidade,False,tuning_cv,0.714070,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.809167
3,KNN,sem_submodalidade,False,test,0.726809,0.725928,0.676821,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:35.312272
4,KNN,submodalidade_agrupada,True,tuning_cv,0.753795,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:43:08.562940
5,KNN,submodalidade_agrupada,True,test,0.752322,0.714884,0.680909,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.216780
6,KNN,submodalidade_agrupada,False,tuning_cv,0.767016,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.218202
7,KNN,submodalidade_agrupada,False,test,0.765706,0.764499,0.711024,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:47:32.638416



🔎 Scenario: submodalidade_engineered
Colunas utilizadas para o cenario 'submodalidade_engineered': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salári

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.709504,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:03:54.190012
1,KNN,sem_submodalidade,True,test,0.720504,0.712963,0.670031,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.808451
2,KNN,sem_submodalidade,False,tuning_cv,0.714070,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.809167
3,KNN,sem_submodalidade,False,test,0.726809,0.725928,0.676821,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:35.312272
4,KNN,submodalidade_agrupada,True,tuning_cv,0.753795,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:43:08.562940
5,KNN,submodalidade_agrupada,True,test,0.752322,0.714884,0.680909,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.216780
6,KNN,submodalidade_agrupada,False,tuning_cv,0.767016,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.218202
7,KNN,submodalidade_agrupada,False,test,0.765706,0.764499,0.711024,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:47:32.638416
8,KNN,submodalidade_engineered,True,tuning_cv,0.722572,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 17:19:14.533930
9,KNN,submodalidade_engineered,True,test,0.722027,0.703553,0.664191,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 17:21:14.074163


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,True,tuning_cv,0.709504,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:03:54.190012
1,KNN,sem_submodalidade,True,test,0.720504,0.712963,0.670031,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.808451
2,KNN,sem_submodalidade,False,tuning_cv,0.714070,NaN,NaN,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:17.809167
3,KNN,sem_submodalidade,False,test,0.726809,0.725928,0.676821,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",2026-06-06 16:04:35.312272
4,KNN,submodalidade_agrupada,True,tuning_cv,0.753795,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:43:08.562940
5,KNN,submodalidade_agrupada,True,test,0.752322,0.714884,0.680909,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.216780
6,KNN,submodalidade_agrupada,False,tuning_cv,0.767016,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:45:36.218202
7,KNN,submodalidade_agrupada,False,test,0.765706,0.764499,0.711024,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 16:47:32.638416
8,KNN,submodalidade_engineered,True,tuning_cv,0.722572,NaN,NaN,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 17:19:14.533930
9,KNN,submodalidade_engineered,True,test,0.722027,0.703553,0.664191,"{'knn__metric': 'manhattan', 'knn__n_neighbors...",2026-06-06 17:21:14.074163
